# Assignment 5 - Reranker and Consolidator

## FP3 Not in Context - Consolidation strategy Limitations
Documents with the answer were retrieved from the database but did not make it into the context for generating an answer. This occurs when many documents are returned from the database and a consolidation process takes place to retrieve the answer.


## Goal
This notebook will address the reranking and consolidation steps of the RAG system

### Reranking
Cross Encoder, MMR (Maximal Marginal Relevance), Reciprocal Rank Fusion


### Consolidation
Possible include an LLM call?


In [1]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia

!pip install rouge-score

In [19]:
import os
import numpy as np
import time
import locale
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')


# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

COHERE_API_KEY = userdata.get('COHERE_API_KEY')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

from rouge_score import rouge_scorer


import os
os.environ["USER_AGENT"] = "RAG_Assignment/v0.1 (davidschaaf@berkeley.edu)"

# 2. Restore your data at the start of a new session
!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_40.tar.gz" -C /content/qdrant_storage/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [3]:
%%capture
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import linear_kernel # dot product
EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
base_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL)

In [29]:
vector_store = QdrantVectorStore(
    client=QdrantClient(path="/content/qdrant_storage"),
    embedding=base_embeddings,
    collection_name="rag_tech_db",
    distance=Distance.DOT
)


In [ ]:
# %%capture
# quantization_config = BitsAndBytesConfig(
#    load_in_4bit=True,
#    bnb_4bit_quant_type="nf4",
#    bnb_4bit_use_double_quant=True,
#    bnb_4bit_compute_dtype=torch.bfloat16
# )

# llm_mistral_model = AutoModelForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-Instruct-v0.3",
#     dtype=torch.float32,
#     device_map='auto',
#     quantization_config=quantization_config
# )

# llm_mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

# mistral_pipe = pipeline(
#     "text-generation",
#     model=llm_mistral_model,
#     tokenizer=llm_mistral_tokenizer,
#     max_new_tokens=1000,
#     temperature=0.6,
#     top_p=0.95,
#     do_sample=True,
#     repetition_penalty=1.2
# )

# mistral_pipe.model.config.pad_token_id = mistral_pipe.model.config.eos_token_id

# mistral_llm_lc = HuggingFacePipeline(pipeline=mistral_pipe)

In [34]:
%%capture
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
marketing_persona = {
    "name": "marketing",
    "description": """This user is a marketer who will ask questions about generative AI in order to better understand the products and the field as a whole.
                      They prefer high level answers that explain concept over technical detail.
                      You will help them find accurate, approved messaging about generative AI features, competitive positioning, and technical capabilities to accelerate content production.""",
}

research_persona = {
    "name": "research",
    "description": """This user is an engineer, who requires detailed technical information when they ask questions.
                      You will help them by writing questions about generative AI concepts, internal system architecture, and implementation details."""
}

llm_template = """Write a question that a {name} professional would ask based on the following context.
{description}

Context:
{context}

Respond only with the question nothing else.
Question:"""
llm_prompt_template = PromptTemplate(template=llm_template, input_variables=["object"])

cohere_chat_model = ChatCohere(cohere_api_key=COHERE_API_KEY)

cohere_chain = llm_prompt_template | cohere_chat_model | StrOutputParser()


In [37]:
import json
import json
import sys
from pathlib import Path
import numpy as np

path = "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_output.json"
records = json.loads(Path(path).read_text())
config = Path(path).stem

# {
#   "persona": "research",
#   "question_score": 28.551249964291134,
#   "question_id": "2507.09477",
#   "recall": [
#     1,
#     0,
#     0,
#     0,
#     0
#   ],
#   "question": "How does the Retrieval-Augmented Decision Transformer architecture enhance the capabilities of LLM agents in research assistance tasks?",
#   "id": "2507.09477",
#   "context_id": "0b6d761a9f0a44e88055cde833633dbc"
# }


In [ ]:
import random
random.seed(42)

PERSONAS = ["research", "marketing"]

def eval_context_persona_reranked(persona, persona_name, record):
    question=record['question']
    q_id=record['question_id']
    document_id=record['id']
    context_id=record['context_id']
    vector_store_result = [doc for doc, score, in vector_store.vector_store.similarity_search_with_score(question, k=20)]
    scores = cross_encoder.predict([(question, doc.page_content) for doc in vector_store_result])
    reranked_docs = [d for score, d in sorted(zip(scores, vector_store_result), key=lambda x: x[0], reverse=True)][:5]

    recall = [1 if x[0].metadata['id'] == document_id else 0 for x in reranked_docs]
    return {
        'persona': persona_name,
        'question_score': question_score,
        'question_id': question_id,
        'recall': recall,
        'question': question,
        'id': document_id,
        'context_id': context_id
    }


points, _ = vector_store.vector_store.client.scroll(
    collection_name=vector_store.vector_store.collection_name,
    limit=1000, with_payload=True)

output = []
research_scores = []
marketing_scores = []
research_recall_1 = []
research_recall_3 = []
research_recall_5 = []
marketing_recall_1 = []
marketing_recall_3 = []
marketing_recall_5 = []

for record in records:
    research_result = eval_context_persona_reranked(research_persona, 'research', record)
    marketing_result = eval_context_persona_reranked(marketing_persona, 'marketing', record)

    output.append(research_result)
    output.append(marketing_result)

    research_scores.append(research_result['question_score'])
    marketing_scores.append(marketing_result['question_score'])

    research_recall_1.append(any(research_result['recall'][:1]))
    research_recall_3.append(any(research_result['recall'][:3]))
    research_recall_5.append(any(research_result['recall'][:5]))
    marketing_recall_1.append(any(marketing_result['recall'][:1]))
    marketing_recall_3.append(any(marketing_result['recall'][:3]))
    marketing_recall_5.append(any(marketing_result['recall'][:5]))



print(f"Average Research Score = {np.mean(research_scores):.3f}")
print(f"Average Marketing Score = {np.mean(marketing_scores):.3f}")
print(f"Average Research Recall@1 = {np.mean(research_recall_1):.3f}")
print(f"Average Research Recall@3 = {np.mean(research_recall_3):.3f}")
print(f"Average Research Recall@5 = {np.mean(research_recall_5):.3f}")
print(f"Average Marketing Recall@1 = {np.mean(marketing_recall_1):.3f}")
print(f"Average Marketing Recall@3 = {np.mean(marketing_recall_3):.3f}")
print(f"Average Marketing Recall@5 = {np.mean(marketing_recall_5):.3f}")